In [98]:
import os
import re
import requests
import asyncio
import json
from typing import Annotated
import dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
# from semantic_kernel.contents import ChatHistory, FunctionCallContent, FunctionResultContent
from semantic_kernel.functions import KernelArguments, kernel_function

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

In [85]:
dotenv.load_dotenv("credentials.env", verbose=True, override=True)

True

In [86]:
import semantic_kernel
print(semantic_kernel.__version__)

1.28.0


In [107]:
def convert_to_json(entity):
    # Convert the result to JSON format
    data = {
        "documents": [
            {            
                "text": entity.text,
                "normalized_text": entity.normalized_text,
                "category": entity.category,
                "subcategory": entity.subcategory,
            }            
        ]
    }
    return json.dumps(data, indent=4)

In [ ]:
class healthanalytics:       
        

    @kernel_function(description="create labels for health text using health analytics")
    def apply_health_labels(self, messages: Annotated[str, "user input messages"]) -> Annotated[str, "Returns labels for the given text."]:
        key = os.environ.get('LANGUAGE_KEY')
        endpoint = os.environ.get('LANGUAGE_ENDPOINT')        
        
        # Authenticate the client using your key and endpoint 
        def authenticate_client():
            ta_credential = AzureKeyCredential(key)
            text_analytics_client = TextAnalyticsClient(
                    endpoint=endpoint, 
                    credential=ta_credential)
            return text_analytics_client

        client = authenticate_client()
        #Patient needs to take 50 mg of ibuprofen.
        poller = client.begin_analyze_healthcare_entities(self.args["messages"])
        result = poller.result()

        docs = [doc for doc in result if not doc.is_error]

        for idx, doc in enumerate(docs):
            j = j + convert_to_json(doc)
        return j
    
    @kernel_function(description="apply labels for ICD10 codes.")
    def apply_icd10_labels(self)-> Annotated[str, "Returns property listings available for lease."]:
        http://icd10api.com/?code=R06.2&desc=short&r=json
        return f"{{\"labels\": [\"label1\", \"label2\"]}}"
    


In [165]:
# Simulate a conversation with the agent
USER_INPUTS = [
    "Patient visited for 99213 was diagnosed R06.2 and E11.65. Type 2 diabetes, wheezing.",     
    "Patient needs to take 50 mg of ibuprofen",  
]
# 

In [166]:
service_id = "health-agent"
kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=service_id))
kernel.add_plugin(healthanalytics(), plugin_name="healthplugin")


# 2. Configure the function choice behavior to auto invoke kernel functions
# so that the agent can automatically execute the menu plugin functions when needed
settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()


In [167]:
# 3. Create the agent
# Create the agent by directly providing the chat completion service
health_agent = ChatCompletionAgent(    
    kernel=kernel,
    name="Assistant",    
    instructions="""You are a health analytics agent. You help labels for health text using health analytics.
                  You can also apply labels for ICD10 codes.                                     
                  You are friendly and helpful. """,
    arguments=KernelArguments(settings=settings),
)

In [168]:
for user_input in USER_INPUTS:
    print(f"User: {user_input}")

User: Patient visited for 99213 was diagnosed R06.2 and E11.65. Type 2 diabetes, wheezing.
User: Patient needs to take 50 mg of ibuprofen


In [169]:
# 4. Create a thread to hold the conversation
# If no thread is provided, a new thread will be
# created and returned with the initial response
thread: ChatHistoryAgentThread = None

In [170]:
for user_input in USER_INPUTS:
        print(f"# User: {user_input}")
        # 5. Invoke the agent for a response
        async for response in health_agent.invoke(messages=user_input, thread=thread):
            print(f"# {response.name}: {response}")
            thread = response.thread

# User: Patient visited for 99213 was diagnosed R06.2 and E11.65. Type 2 diabetes, wheezing.


Function failed. Error: key must be a string.
Error invoking function healthplugin-apply_health_labels: key must be a string..
Traceback (most recent call last):
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\kernel.py", line 422, in _inner_auto_function_invoke_handler
    result = await context.function.invoke(context.kernel, context.arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 258, in invoke
    raise e
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 250, in invoke
    await stack(function_context)
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function_from_method.py", line 103, in _

# Assistant: It looks like there was an issue with applying health labels to the provided text. However, ICD-10 codes have been identified.

The ICD-10 codes associated with the patient's visit are:
- **R06.2**: Wheezing
- **E11.65**: Type 2 diabetes mellitus with hyperglycemia

For more precise labeling and documentation, it may be helpful to reformat the input or provide additional context. If you need further assistance, please let me know!
# User: Patient needs to take 50 mg of ibuprofen


Function failed. Error: key must be a string.
Error invoking function healthplugin-apply_health_labels: key must be a string..
Traceback (most recent call last):
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\kernel.py", line 422, in _inner_auto_function_invoke_handler
    result = await context.function.invoke(context.kernel, context.arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 258, in invoke
    raise e
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 250, in invoke
    await stack(function_context)
  File "c:\Users\nileshjoshi\git\aoai-poc-hub\common\envs\aoai_poc_e5\Lib\site-packages\semantic_kernel\functions\kernel_function_from_method.py", line 103, in _

# Assistant: It appears there was another issue with applying health labels to the provided text. However, we have successfully identified the ICD-10 code labels.

For the medication prescription:
- The patient needs to take 50 mg of ibuprofen.

Unfortunately, specific ICD-10 coding for medication prescriptions isn't typically applicable unless associated directly with a diagnosis.

If you need any specific health labels or additional help, please let me know!


In [95]:
# 6. Cleanup: Clear the thread
await thread.delete() if thread else None